In [1]:
# intorduction about langchain
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()

True

In [2]:
# step 1
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke('hi').content

'Hello! How can I assist you today?'

In [3]:
import langchain
import langchain_openai
import langchain_core
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
print('version of langchain: ',langchain.__version__)
print('version of langchain_core: ',langchain_core.__version__)

version of langchain:  1.2.17
version of langchain_core:  1.3.2


In [4]:
result = llm.invoke('hi')

In [5]:
result.response_metadata

{'token_usage': {'completion_tokens': 9,
  'prompt_tokens': 8,
  'total_tokens': 17,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-4o-mini-2024-07-18',
 'system_fingerprint': 'fp_845d726e38',
 'id': 'chatcmpl-DbyypTy9hgffgryRx77JcX70o3kId',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

In [8]:
from langchain_core.prompts import ChatPromptTemplate

simple_prompt = ChatPromptTemplate.from_template(
    'Reply in one sentence: {questions}'
)

simple_prompt.invoke({'questions': 'What is the capital of France?'})

ChatPromptValue(messages=[HumanMessage(content='Reply in one sentence: What is the capital of France?', additional_kwargs={}, response_metadata={})])

In [9]:
classifier_template = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert support email classifier for a B2B SaaS company.'),
    ('human',  'Classify this email.\nSubject: {subject}\nBody: {body}\n\nReturn ONLY: Category | Urgency')
])


In [14]:
# chain 
# manauly --> prompt --llm -->output --> clean 
# langchain --> chain ( pipeline ) --> input or prompt --> operation ? 
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

classify_chain = classifier_template | llm | parser

In [11]:
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body'   : "Our team can't login. We paid last week. Board demo in 3 hours."
}

In [15]:
classify_chain.invoke(email)

'Category: Account Access | Urgency: High'

In [16]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

emails[0]

{'subject': 'Payment failed', 'body': 'Charged twice this month.'}

In [17]:
classify_chain.invoke(emails[0])

'Billing | High'

In [18]:
import time
start = time.time()
loop_result = []
for email in emails:
    loop_result.append(classify_chain.invoke(email))

loop_time = time.time()-start
print(loop_time)    

10.27809190750122


In [19]:
batch_results = classify_chain.batch(emails,
                                    config={'max_concurrency':3})

In [20]:
batch_results

['Category: Billing | Urgency: High',
 'Category: Technical Support | Urgency: High',
 'Feature Request | Low',
 'Category: Spam | Urgency: Low',
 'Billing | Medium']

In [ ]:
# issue ?? plain python --> ourself
# 1000 emails. --> LLM --> Ratelimit error --> api key error --> error handle ?? 
# langchain
# Rate limit --> within 1 sec if you are calling LLM 4-5 times --> reject request
# congestion s--> wait error --> 1s,2s,4s,8s wait 

In [21]:
classify_chain.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True
)

RunnableRetry(bound=ChatPromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert support email classifier for a B2B SaaS company.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={}, template='Classify this email.\nSubject: {subject}\nBody: {body}\n\nReturn ONLY: Category | Urgency'), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 